# 03 · Conditional VAE — nutrient-profile generation
Generate nutrient profiles conditioned on a target Nutri-Score grade.
Improves on a plain VAE (the reference project's future work).


In [ ]:
# Make src/ importable whether on Colab or local
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[0] if (pathlib.Path.cwd().name=='notebooks') else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT/'src'))
from nutricoach import config as C


In [ ]:
import numpy as np, torch, joblib
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from nutricoach import data, cvae


In [ ]:
df = data.premium_subset(data.load_clean())
X = df[C.NUTRIENT_FEATURES].astype(float).values
grades = df[C.CLASSIFICATION_TARGET].str.lower().map({g:i for i,g in enumerate(C.NUTRISCORE_GRADES)}).values
scaler = StandardScaler().fit(X)
Xs = scaler.transform(X)
C_onehot = np.eye(len(C.NUTRISCORE_GRADES))[grades]


In [ ]:
ds = TensorDataset(torch.tensor(Xs, dtype=torch.float32), torch.tensor(C_onehot, dtype=torch.float32))
loader = DataLoader(ds, batch_size=128, shuffle=True)
model = cvae.CVAE(); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(50):
    for xb, cb in loader:
        recon, mu, logvar = model(xb, cb)
        loss, *_ = cvae.loss_fn(recon, xb, mu, logvar, beta=1.0)
        opt.zero_grad(); loss.backward(); opt.step()
print('trained')


In [ ]:
torch.save(model.state_dict(), cvae.CVAE_PATH)
joblib.dump(scaler, cvae.SCALER_PATH)
cvae.generate_profile('a', n=3)  # sanity check
